# MI vs Magnitude — automated comparison
Runs prune → finetune → sample → FID for each method, then prints a table.

## Setup

In [ ]:
!git clone --branch mipp https://github.com/elliotcanter11/Diff-Pruning.git

In [ ]:
%cd Diff-Pruning/

In [ ]:
!pip install -r requirements.txt

In [ ]:
!python tools/extract_cifar10_hug.py --output data

In [ ]:
!bash tools/convert_cifar10_ddpm_ema.sh

In [ ]:
# real-image FID statistics (run once)
!python fid_score.py --save-stats data/cifar10_images run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256

## Config + run
`FID_SAMPLES` is the main time/noise knob — more = slower but less noisy. Comment methods out of the dict to skip them.

Each MI prune also prints an **overlap-vs-magnitude** diagnostic (Jaccard of the pruned set + Spearman rank corr). ~1.0 means it's cutting the same filters as magnitude; lower means the MI criterion is genuinely choosing differently.

In [ ]:
import os, subprocess, re

RATIO       = 0.3      # pruning ratio (0.5 stresses harder but FID gets noisy)
ITERS       = 1000     # finetuning steps
WORKERS     = 12
FID_SAMPLES = 10000    # 50000 for a final number; lower for a quick look
DDIM_STEPS  = 100      # 50 ~halves sampling time; fair since all methods match

# prune_ddpm_cifar10_mi.sh args: ratio  w_output  w_adjacency
METHODS = {
    'magnitude'    : f'bash scripts/prune_ddpm_cifar10.sh {RATIO}',
    'mi_adjacency' : f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 0.0 1.0',
    'mi_output'    : f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 1.0 0.0',
    # 'mi_combined'  : f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 1.0 1.0',
}

def run(cmd):
    print('\n$', cmd, flush=True)
    assert os.system(cmd) == 0, f'FAILED: {cmd}'

def fid():
    out = subprocess.run(
        'python fid_score.py run/sample/ddpm_cifar10_pruned run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256',
        shell=True, capture_output=True, text=True).stdout
    print(out.strip().splitlines()[-1] if out.strip() else '(no fid output)')
    m = re.search(r'FID:\s*([\d.]+)', out)
    return float(m.group(1)) if m else float('nan')

results = {}
for name, prune_cmd in METHODS.items():
    print('\n' + '='*60 + f'\n{name}\n' + '='*60, flush=True)
    run('rm -rf run/pruned/ddpm_cifar10_pruned '
        'run/finetuned/ddpm_cifar10_pruned_post_training run/sample/ddpm_cifar10_pruned')
    run(prune_cmd)
    run(f'bash scripts/finetune_ddpm_cifar10.sh {ITERS} {WORKERS}')
    run(f'python ddpm_sample.py --output_dir run/sample/ddpm_cifar10_pruned '
        f'--batch_size 256 --total_samples {FID_SAMPLES} --ddim_steps {DDIM_STEPS} '
        f'--pruned_model_ckpt run/finetuned/ddpm_cifar10_pruned_post_training/pruned/unet_ema_pruned.pth '
        f'--model_path run/finetuned/ddpm_cifar10_pruned_post_training --skip_type uniform')
    results[name] = fid()
    print(f'\n>>> {name} FID = {results[name]:.2f}')


## Results

In [ ]:
print(f'ratio={RATIO}   finetune={ITERS} steps   fid_samples={FID_SAMPLES}\n')
for name, f in sorted(results.items(), key=lambda x: x[1]):
    print(f'{name:14s} FID {f:.2f}')
